# Day 66: Transfer Learning
## Stand on the Shoulders of Giants — Pre-trained Models

---

# PART 1: THEORY

## 1. The Problem with Training from Scratch

Training a CNN on ImageNet (1.2 million images, 1000 classes) from scratch takes:
- **Days to weeks** on multiple GPUs
- **Thousands of dollars** in cloud compute
- Carefully tuned hyperparameters

**You don't have that time or money. And you don't need to!**

## 2. Transfer Learning — The Smart Shortcut

**Core Idea:** A CNN trained on ImageNet has already learned to detect edges, shapes, textures, and objects. These features are useful for ANY image task.

**The process:**
1. Take a pre-trained model (trained on ImageNet)
2. Chop off the final classification layer
3. Add YOUR classification layers on top
4. Train only the new layers (freeze the pre-trained base)

```
[Pre-trained CNN]  ->  [Your Dense Layers]  ->  [Your Output]
   (FROZEN)              (TRAINABLE)             (TRAINABLE)
   "Feature Extractor"   "Custom Classifier"
```

**This works because:** The early layers detect universal features (edges, curves) while later layers detect task-specific features (dog ears vs cat ears). You keep the universal features and retrain the task-specific part.

## 3. Popular Pre-trained Models

| Model | Size | Top-1 Acc | Best For |
|-------|------|-----------|----------|
| **MobileNetV2** | 14 MB | 71.3% | Mobile, fast inference, learning |
| **ResNet50** | 98 MB | 74.9% | General purpose, good accuracy |
| **VGG16** | 528 MB | 71.3% | Teaching (simple architecture) |
| **EfficientNetB0** | 29 MB | 77.1% | Best accuracy/size ratio |
| **InceptionV3** | 92 MB | 77.9% | Production systems |

## 4. Fine-Tuning — The Next Level

After training your classifier layers, you can **unfreeze** the last few layers of the pre-trained model and train them with a VERY low learning rate. This adapts the pre-trained features to your specific data.

---

# PART 2: PRACTICAL

## 5. Setup and Load Data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, applications

# Use CIFAR-10
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# Normalize
X_train_n = X_train.astype('float32') / 255.0
X_test_n = X_test.astype('float32') / 255.0

# Resize to 224x224 (pre-trained models expect this size)
X_train_r = tf.image.resize(X_train_n, (224, 224))
X_test_r = tf.image.resize(X_test_n, (224, 224))

print(f"Original CIFAR: {X_train.shape[1]}x{X_train.shape[2]}")
print(f"Resized: {X_train_r.shape[1]}x{X_train_r.shape[2]} -> Ready for pre-trained models!")


In [ ]:
# Show resized vs original
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i in range(3):
    axes[i].imshow(X_train[i])
    axes[i].set_title(f'Original 32x32 — {classes[y_train[i][0]]}')
    axes[i].axis('off')
plt.suptitle('CIFAR-10 (32x32) -> Will be resized to 224x224', fontsize=14)
plt.tight_layout()
plt.show()


## 6. Build Transfer Learning Model

In [ ]:
# Load pre-trained MobileNetV2 (no top layer)
base_model = applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,       # Discard original 1000-class classifier
    weights='imagenet'       # Load weights pre-trained on ImageNet
)

# FREEZE the base — don't destroy pre-trained knowledge!
base_model.trainable = False

# Build complete model
model = keras.Sequential([
    # Preprocessing for MobileNetV2 (expects [-1, 1] range)
    layers.Rescaling(scale=1./127.5, offset=-1, input_shape=(224, 224, 3)),

    # Pre-trained feature extractor
    base_model,

    # Pool feature maps to a single vector
    layers.GlobalAveragePooling2D(),

    # Custom classifier (only these are trained!)
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

# Count trainable vs total
total = model.count_params()
trainable = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"\nTotal parameters:     {total:,}")
print(f"Trainable parameters:   {trainable:,}")
print(f"Frozen parameters:      {total - trainable:,}")
print(f"\nWe only train {trainable/total*100:.1f}% of the parameters!")


## 7. Train and Evaluate

In [ ]:
# Train for 5 epochs (uses cached features after first epoch — fast!)
history = model.fit(X_train_r, y_train, epochs=5, batch_size=32,
                    validation_split=0.1, verbose=1)

test_loss, test_acc = model.evaluate(X_test_r, y_test, verbose=0)
print(f"\nTransfer Learning Accuracy: {test_acc:.4f} ({test_acc*100:.1f}%)")


In [ ]:
# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['accuracy'], label='Train', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Transfer Learning Training')
axes[0].legend()
axes[0].grid(True)
axes[1].plot(history.history['loss'], label='Train', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('Transfer Learning Loss')
axes[1].legend()
axes[1].grid(True)
plt.tight_layout()
plt.show()


## 8. Fine-Tuning — Unlock the Last Layers

In [ ]:
# Unfreeze the base model
base_model.trainable = True

# Freeze all layers EXCEPT the last 20
fine_tune_at = len(base_model.layers) - 20
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Base has {len(base_model.layers)} layers")
print(f"First {fine_tune_at} layers are FROZEN")
print(f"Last {len(base_model.layers) - fine_tune_at} layers are TRAINABLE")

# Recompile with VERY low learning rate for fine-tuning
model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Fine-tune for a few epochs
history_ft = model.fit(X_train_r, y_train, epochs=3, batch_size=32,
                        validation_split=0.1, verbose=1)

ft_loss, ft_acc = model.evaluate(X_test_r, y_test, verbose=0)
print(f"\nBefore fine-tuning: {test_acc:.4f}")
print(f"After fine-tuning:  {ft_acc:.4f}")
print(f"Improvement:         +{(ft_acc - test_acc)*100:.1f}%")


## 9. Compare Models on Same Data

In [ ]:
# Comparison table
print("Model                         | Accuracy | Params")
print("-" * 55)

# Transfer learning
print(f"Transfer Learning (MobileNet)  |  {ft_acc:.3f}   | {model.count_params():,}")

# CNN from scratch (Day 65)
cnn = keras.Sequential([
    layers.Conv2D(32, 3, activation='relu', padding='same', input_shape=(32,32,3)),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu', padding='same'),
    layers.MaxPooling2D(2),
    layers.Conv2D(128, 3, activation='relu', padding='same'),
    layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation='softmax')
])
cnn.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.fit(X_train_n, y_train, epochs=5, validation_split=0.1, verbose=0)
_, cnn_acc = cnn.evaluate(X_test_n, y_test, verbose=0)
print(f"CNN from scratch (5 epochs)    |  {cnn_acc:.3f}   | {cnn.count_params():,}")

# Simple Dense
dense = keras.Sequential([
    layers.Flatten(input_shape=(32,32,3)),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])
dense.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
dense.fit(X_train_n, y_train, epochs=5, validation_split=0.1, verbose=0)
_, dense_acc = dense.evaluate(X_test_n, y_test, verbose=0)
print(f"Dense Network (5 epochs)       |  {dense_acc:.3f}   | {dense.count_params():,}")

print(f"\nTransfer learning wins — higher accuracy with less training!")


---

# PART 3: EXERCISES

In [ ]:
# Exercise 1: Try a different pre-trained model (ResNet50)
from tensorflow.keras import applications

base_resnet = applications.ResNet50(input_shape=(224,224,3), include_top=False, weights='imagenet')
base_resnet.trainable = False

resnet_model = keras.Sequential([
    layers.Rescaling(scale=1./127.5, offset=-1, input_shape=(224,224,3)),
    base_resnet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation='softmax')
])
resnet_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
resnet_model.fit(X_train_r[:2000], y_train[:2000], epochs=3, validation_split=0.2, verbose=0)
_, rn_acc = resnet_model.evaluate(X_test_r, y_test, verbose=0)
print(f"MobileNetV2: {ft_acc:.3f} | ResNet50: {rn_acc:.3f}")
print("Different models have different strengths!")


In [ ]:
# Exercise 2: Try with only 50 training images per class
n_per_class = 50
indices = []
for c in range(10):
    c_indices = np.where(y_train.flatten() == c)[0][:n_per_class]
    indices.extend(c_indices)
X_small = X_train_r[np.array(indices)]
y_small = y_train[np.array(indices)]

# Build a quick model with the same base
quick_model = keras.Sequential([
    layers.Rescaling(scale=1./127.5, offset=-1, input_shape=(224, 224, 3)),
    base_model, layers.GlobalAveragePooling2D(),
    layers.Dense(10, activation='softmax')
])
quick_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
quick_model.fit(X_small, y_small, epochs=5, validation_split=0.2, verbose=0)
_, acc_small = quick_model.evaluate(X_test_r, y_test, verbose=0)
print(f"With {n_per_class} images/class: {acc_small:.3f}")
print(f"With full dataset: {ft_acc:.3f}")
print("Transfer learning works even with VERY small datasets!")


## Key Takeaways

- **Transfer learning** = use pre-trained model + add your classifier
- **Freeze** the base during initial training (don't destroy pre-trained weights)
- **Fine-tune** with very low learning rate to adapt to your data
- Pre-trained models have learned universal image features from ImageNet
- Works with **small datasets** (even 50-100 images per class!)
- **MobileNetV2** is great for learning (small, fast)
- Transfer learning is what industry uses — almost nobody trains from scratch

**Tomorrow:** RNNs & LSTMs — understanding sequential data!